In [1]:
import os
import numpy as np
import pandas as pd
from functools import reduce
from tqdm import tqdm   # ✅ 引入进度条库
dir = "/project/model_share_remote/other_data/barra_factors/"
fac_list = os.listdir(dir)


In [2]:
dfs = []
for fac in tqdm(fac_list, desc="Processing factors"):  # ✅ 加 tqdm 包裹
    # 读入
    df = pd.read_feather(os.path.join(dir, fac))

    # 宽转长：index=日期；其余列为股票代码；值为该因子原始值
    df_long = df.melt(id_vars=["index"], var_name="Code", value_name="value")

    # —— 截面标准化（对每个日期的横截面做 z-score）——
    g = df_long.groupby("index")["value"]
    z = (df_long["value"] - g.transform("mean")) / g.transform("std")

    # 处理异常与缺失：std 为 0、全空、或原始 NaN 都变成 0
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0)
    z = z.infer_objects(copy=False)
    df_long["value"] = z

    # 给列名加上因子名（去掉扩展名）
    fac_name = os.path.splitext(fac)[0]
    df_long = df_long.rename(columns={"value": fac_name})

    dfs.append(df_long)

# 按照 date, Code 合并
df_all = reduce(lambda left, right: pd.merge(left, right, on=["index","Code"], how="outer"), dfs)

# 最后设定 index
df_all = df_all.set_index(["index", "Code"]).sort_index()


Processing factors:  14%|█▍        | 2/14 [00:05<00:30,  2.55s/it]/tmp/ipykernel_259113/669458674.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  z = z.replace([np.inf, -np.inf], np.nan).fillna(0)
Processing factors:  21%|██▏       | 3/14 [00:12<00:52,  4.75s/it]/tmp/ipykernel_259113/669458674.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  z = z.replace([np.inf, -np.inf], np.nan).fillna(0)
Processing factors:  36%|███▌      | 5/14 [00:23<00:43,  4.80s/it]/tmp/ipykernel_259113/669458674.py:14: FutureWarning: Downcasting 

In [3]:
df_all = df_all.reset_index()

In [ ]:
# 1. 先得到所有的唯一日期，并按顺序排序
dates = sorted(df_all["index"].unique())
# 2. 构造日期映射：当前日期 -> 下一交易日
date_shift_map = {dates[i]: dates[i+1] for i in range(0, len(dates)-1, 1)}
# 3. 应用映射，把没有下一日的日期丢弃
df_all["index_shifted"] = df_all["index"].map(date_shift_map)
# 4. 去掉 NaN（因为最后一天无法平移）
df_all = df_all.dropna(subset=["index_shifted"])
df_all["index_shifted"] = df_all["index_shifted"].astype(int)
# 5. 把平移后的日期列改名为 index，丢掉旧的 index 列
df_all = df_all.drop(columns=["index"]).rename(columns={"index_shifted": "date"})
df_all = df_all.set_index(["date", "Code"]).sort_index()


In [8]:
df_reset = df_all.reset_index()
df_reset['date'] = df_reset['date'].astype(str)
df_reset.to_feather("/home/user96/score_file/barra.fea")

In [9]:
import pandas as pd 
data = pd.read_feather("/home/user96/score_file/barra.fea")
data

,date,Code,fac_daily_std,fac_hist_sigma,fac_stoq,fac_strev,fac_rs,fac_atvr,fac_beta,fac_ind_mom,fac_stoa,fac_midcap,fac_lncap,fac_stom,fac_cmra,fac_alpha
0,20150106,000001,0.210597,-1.754034,0.000000,1.864626,0.797525,-0.375713,3.217171,0.368420,-0.745109,-0.162115,3.264595,0.0,0.223643,-0.629622
1,20150106,000002,0.463353,-1.082781,0.000000,1.421173,0.664237,-0.379659,3.006551,1.454012,-0.770846,-0.265088,3.185695,0.0,1.419239,0.104218
2,20150106,000004,0.595587,0.308797,0.000000,-1.859777,-0.519751,0.089653,-0.588544,-0.851211,0.562174,1.470046,-1.171418,0.0,0.289096,-0.734140
3,20150106,000005,0.334751,-0.094305,0.000000,0.437077,0.703368,0.087618,-0.258487,-0.768492,0.233000,-0.112478,-0.201613,0.0,0.257646,0.988615
4,20150106,000006,-0.617548,-0.616793,0.000000,0.202385,0.027721,-0.168664,0.462539,-0.385078,-0.144760,-0.965864,0.658955,0.0,0.065079,0.292400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13722205,20241231,873703,0.638295,0.607385,0.932276,-1.538147,-0.289008,0.226608,1.093521,-1.521737,1.099162,2.430542,-1.649100,0.0,0.121643,-0.599797
13722206,20241231,873706,0.917588,0.595556,1.791069,-1.778787,0.000000,1.392617,0.762055,-1.282595,1.538650,4.123387,-2.306905,0.0,0.693763,-0.321716
13722207,20241231,873726,2.063615,2.096297,1.705474,-1.104475,-0.087223,0.859045,1.365002,-2.008866,1.453028,3.415155,-2.046014,0.0,0.559000,-0.500927
13722208,20241231,873806,1.958989,1.678385,1.888108,-2.516094,0.000000,1.560417,2.303938,-1.798912,2.038843,1.480310,-1.212757,0.0,0.970428,-0.177213
